# Two teams reported "the lift". The numbers differ by a factor of eight.

Neither team made an error. One measured the effect of 100 USD against nothing; the other of
200 against 100. One reported the season total; the other the weekly average. One weighted
the region by its own soil mix; the other by the national one. Every number is correct and
every one answers a different question, and none of the four questions was written down —
because in the parent repo most of these choices lived in a variable name.

An `Estimand` is a complete transferability key. Two quantities are the same quantity iff all
eight facets match:

| Facet | Field(s) |
|---|---|
| `quantity` | `quantity` — contrast, marginal, ratio, elasticity, area |
| `intervention` | `treatment`, `intervention`, `reference` |
| `outcome` | `outcome` |
| `population` | `population` |
| `window` | `window` |
| `level` | `level` — unit of analysis + interference model |
| `conditioning` | `conditioning` |
| `dimension` | `dimension` — derived, asserted against the declaration |

Nothing about an estimand may be implicit in the model that produced it.

In [ ]:
import numpy as np

from axiom.core import D, DimensionError, Intervention, Outcome, Population, Spec, TimeWindow, Treatment
from axiom.estimands import FACETS, Estimand, Facet, Level, Quantity, QuantityKind, derived_dimension

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, compare, curve_band, mark_x

enable();  # every axiom result renders itself from here on

In [ ]:
fertilizer = Treatment(name="fertilizer", dimension=D.currency, unit="USD")
yield_total = Outcome(name="yield_total", dimension=D.outcome, unit="kg", aggregation="sum")
north = Population(name="north", strata={"soil": {"clay": 0.3, "loam": 0.7}})
season = TimeWindow(start=0, stop=8, basis="cumulative")

lift = Estimand(
    name="lift_at_100",
    quantity=Quantity(kind="contrast"),
    treatment=fertilizer,
    intervention=Intervention(doses={"fertilizer": 100.0}, version="granular"),
    reference=Intervention(doses={"fertilizer": 0.0}, version="granular"),
    outcome=yield_total,
    population=north,
    window=season,
    level=Level(unit="cluster", interference="none"),
    conditioning=(),
    dimension=D.outcome,
    description="season-total yield lift from 100 USD of granular fertilizer vs none, north region",
)
print(lift.content_hash()[:16])

## The four numbers from the top of this notebook

One response curve — saturating, as they are — and one true state of the world. Below, the
quantities four honest analysts would report from it, all called "the lift".

In [ ]:
def response(dose, beta=4.2, k=90.0, s=1.6):
    """Season-total yield from a dose, per week, for one soil type."""
    x = np.asarray(dose, dtype=float) / k
    return beta * x**s / (1 + x**s)


soil_multiplier = {"clay": 0.6, "loam": 1.3}     # the same dose does less on clay
mix = lambda weights: sum(soil_multiplier[s] * w for s, w in weights.items())
north_mix, national_mix = mix(north.strata["soil"]), mix({"clay": 0.5, "loam": 0.5})

grid = np.linspace(0, 300, 200)
fig = curve_band(
    grid, response(grid) * season.length * north_mix,
    label="season total",
    title="One curve",
    subtitle="season-total yield against fertilizer spend — the shared truth everybody is reporting from",
    x_title="fertilizer (USD)", y_title="yield (kg)",
)
mark_x(fig, 100.0, text="100 USD")
mark_x(fig, 200.0, text="200 USD", right=True)
caption(fig, "Nothing below is a disagreement about this curve. Every number comes off it.")

In [ ]:
per_week = lambda a, b: response(b) - response(a)
readings = {
    "100 vs 0, season total": per_week(0.0, 100.0) * season.length * north_mix,
    "200 vs 100, season total": per_week(100.0, 200.0) * season.length * north_mix,
    "100 vs 0, per week": per_week(0.0, 100.0) * north_mix,
    "100 vs 0, national soil mix": per_week(0.0, 100.0) * season.length * national_mix,
}
fig = compare(
    list(readings), list(readings.values()),
    highlight="100 vs 0, season total",
    value_fmt="{:.1f}",
    title="…and four things called 'the lift'",
    subtitle="each one correct, each one a different estimand, all of them 'the effect of fertilizer'",
    x_title="kg",
)
caption(fig, "Top to bottom: a different reference dose, a different window basis, a "
             "different population weighting. Three facets out of eight, and the largest "
             "reading is eight times the smallest. The declaration above pins all eight, so "
             "a number cannot be quoted against a question it did not answer.")

## The derived dimension is asserted

`derived_dimension(kind, outcome, dose)` is the table; the class refuses a declaration that
disagrees with it. A contrast is in outcome units, a marginal effect is outcome per dose, and
an elasticity is a pure number — so a declaration that says otherwise is a claim about the
quantity that the quantity itself contradicts.

In [ ]:
kinds: list[QuantityKind] = ["contrast", "marginal", "ratio", "elasticity", "area"]
table(
    [[kind, str(derived_dimension(kind, D.outcome, D.currency))] for kind in kinds],
    headers=("kind", "dimension"),
)

try:
    lift.model_copy(update={"dimension": D.currency}).model_validate(lift.model_copy(update={"dimension": D.currency}).to_dict())
except DimensionError as e:
    print("refused:", e)

## Other functionals

A marginal effect needs no reference; a ratio does. The intervention must set the treatment
the quantity is about.

In [ ]:
mroi = lift.model_copy(update={
    "name": "marginal_at_100",
    "quantity": Quantity(kind="marginal"),
    "reference": None,
    "dimension": D.outcome / D.currency,
})
print(mroi.name, mroi.dimension)

try:
    Estimand.model_validate({**lift.to_dict(), "quantity": {"kind": "ratio", "scale": "natural"}, "reference": None, "dimension": (D.outcome / D.currency).to_dict()})
except ValueError as e:
    print("refused:", str(e).splitlines()[-1].strip())

## Facets are inspectable

`FACETS` is the closed list; `facet()` returns what `transfer_to` compares; `differing_facets`
is the typed diff between two declarations. "These two numbers are not comparable" becomes a
list of field names rather than a hunch.

In [ ]:
print(FACETS)
f: Facet = "intervention"
print(lift.facet(f))
print(lift.differing_facets(mroi))

## Serialization

An estimand is a `Spec`: it round-trips and its hash is its identity. Two producers — an
experiment, a fitted surface, a meta-analysis — that declare the same facets produce the same
hash, which is exactly what makes their results comparable, and what lets a meta-analysis
refuse to pool two studies that measured different things.

In [ ]:
print(Spec.from_json(lift.to_json()) == lift)
print(lift.to_json(indent=2)[:300], "...")

## What this bought you

The argument at the top of this notebook cannot happen. Two numbers either carry the same
eight facets — in which case they are the same quantity and their hashes agree — or they
differ in named ways, and `02-transfer-plans.ipynb` is what it costs to move between them.